# 03 - Model Development

## هدف نوت‌بوک

در این نوت‌بوک مدل‌های یادگیری ماشین برای پیش‌بینی قصد تداوم استفاده از یادگیری الکترونیکی توسعه داده می‌شوند.

### Development Dataset
Dataset B

### Predictors
- Confirmation (CON_score)
- Satisfaction (SAT_score)

### Target
- Continuance Intention (CI_score)

مدل اصلی:

CON_score + SAT_score → CI_score



In [1]:
# در این سلول کتابخانه‌های موردنیاز و مسیر فایل Model-Ready مربوط به Dataset B به‌صورت خودکار تعریف می‌شوند.

from pathlib import Path
import os
import pandas as pd
import numpy as np


# مسیر فعلی اجرای Notebook
CURRENT_DIR = Path.cwd().resolve()


# شناسایی خودکار ریشه پروژه
if CURRENT_DIR.name == "notebooks":

    PROJECT_ROOT = CURRENT_DIR.parent

elif (
    (CURRENT_DIR / "notebooks").exists()
    and (CURRENT_DIR / "data").exists()
):

    PROJECT_ROOT = CURRENT_DIR

else:

    raise FileNotFoundError(
        "Project root could not be detected. "
        "Please run this notebook from the project folder or the notebooks folder."
    )


PROCESSED_FOLDER = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed"
)


B_MODEL_READY_PATH = os.path.join(
    PROCESSED_FOLDER,
    "dataset_B_model_ready.csv"
)


print("Project root:")
print(PROJECT_ROOT)

print("\nDataset B path:")
print(B_MODEL_READY_PATH)

print("\nFile exists:")
print(os.path.exists(B_MODEL_READY_PATH))

Project root:
F:\E_Learning_Continuance_ML

Dataset B path:
F:\E_Learning_Continuance_ML\data\processed\dataset_B_model_ready.csv

File exists:
True


In [2]:
# در این سلول فایل Model-Ready مربوط به Dataset B بارگذاری و ساختار آن بررسی می‌شود.

df_B_model = pd.read_csv(
    B_MODEL_READY_PATH
)

print("Dataset B shape:", df_B_model.shape)

print("\nColumns:")
print(df_B_model.columns.tolist())

print("\nFirst five rows:")
display(df_B_model.head())

Dataset B shape: (368, 8)

Columns:
['Dataset', 'CON_score', 'SAT_score', 'CI_score', 'Straight_Line_Flag', 'Very_Fast_Flag', 'High_Risk_Flag', 'Strict_Duplicate_Flag']

First five rows:


,Dataset,CON_score,SAT_score,CI_score,Straight_Line_Flag,Very_Fast_Flag,High_Risk_Flag,Strict_Duplicate_Flag
0,B,4.333333,4.666667,5.000000,False,False,False,False
1,B,4.333333,4.333333,3.000000,False,False,False,False
2,B,3.333333,2.666667,2.666667,False,False,False,False
3,B,4.666667,5.000000,4.666667,False,False,False,False
4,B,2.000000,3.000000,3.333333,False,False,False,False


In [3]:
# در این سلول دو متغیر CON و SAT به‌عنوان ورودی مدل و CI به‌عنوان متغیر هدف تعریف می‌شوند.

feature_columns = [
    "CON_score",
    "SAT_score"
]

target_column = "CI_score"

X = df_B_model[
    feature_columns
].copy()

y = df_B_model[
    target_column
].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nFeatures:")
display(X.head())

print("\nTarget:")
display(y.head())

X shape: (368, 2)
y shape: (368,)

Features:


,CON_score,SAT_score
0,4.333333,4.666667
1,4.333333,4.333333
2,3.333333,2.666667
3,4.666667,5.000000
4,2.000000,3.000000



Target:


0    5.000000
1    3.000000
2    2.666667
3    4.666667
4    3.333333
Name: CI_score, dtype: float64

In [4]:
# در این سلول Dataset B به داده آموزشی و داده آزمون داخلی با نسبت 80 به 20 تقسیم می‌شود.

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

Training features: (294, 2)
Testing features: (74, 2)
Training target: (294,)
Testing target: (74,)


In [5]:
# در این سلول یک مدل Baseline ساخته می‌شود که برای همه افراد مقدار میانگین CI در داده آموزشی را پیش‌بینی می‌کند.

baseline_value = y_train.mean()

baseline_predictions = np.full(
    shape=len(y_test),
    fill_value=baseline_value
)

print("Mean CI in training data:")
print(round(baseline_value, 3))

print("\nFirst 10 baseline predictions:")
print(
    baseline_predictions[:10]
)

Mean CI in training data:
3.974

First 10 baseline predictions:
[3.9739229 3.9739229 3.9739229 3.9739229 3.9739229 3.9739229 3.9739229
 3.9739229 3.9739229 3.9739229]


In [6]:
# در این سلول عملکرد Baseline با معیارهای MAE، RMSE و R² روی داده آزمون داخلی محاسبه می‌شود.

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

baseline_mae = mean_absolute_error(
    y_test,
    baseline_predictions
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        baseline_predictions
    )
)

baseline_r2 = r2_score(
    y_test,
    baseline_predictions
)

print("Baseline performance:")
print("MAE :", round(baseline_mae, 4))
print("RMSE:", round(baseline_rmse, 4))
print("R²  :", round(baseline_r2, 4))

Baseline performance:
MAE : 0.646
RMSE: 0.8374
R²  : -0.0077


In [7]:
# در این سلول مدل Linear Regression با استفاده از CON و SAT روی داده آموزشی Dataset B آموزش داده می‌شود.

from sklearn.linear_model import LinearRegression

linear_model = LinearRegression()

linear_model.fit(
    X_train,
    y_train
)

linear_predictions = linear_model.predict(
    X_test
)

print("Linear Regression trained successfully.")

print("\nFirst 10 predictions:")
print(
    np.round(
        linear_predictions[:10],
        3
    )
)

Linear Regression trained successfully.

First 10 predictions:
[3.9   3.23  3.23  4.054 4.821 4.026 4.026 3.495 4.026 3.23 ]


In [8]:
# در این سلول عملکرد Linear Regression با معیارهای MAE، RMSE و R² روی داده آزمون داخلی ارزیابی می‌شود.

linear_mae = mean_absolute_error(
    y_test,
    linear_predictions
)

linear_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        linear_predictions
    )
)

linear_r2 = r2_score(
    y_test,
    linear_predictions
)

print("Linear Regression performance:")

print(
    "MAE :",
    round(linear_mae, 4)
)

print(
    "RMSE:",
    round(linear_rmse, 4)
)

print(
    "R²  :",
    round(linear_r2, 4)
)

Linear Regression performance:
MAE : 0.3254
RMSE: 0.5315
R²  : 0.5941


In [9]:
# در این سلول ضرایب مدل Linear Regression استخراج می‌شوند تا نقش CON و SAT در پیش‌بینی CI بررسی شود.

linear_coefficients = pd.DataFrame({
    "Feature": feature_columns,
    "Coefficient": linear_model.coef_
})

linear_coefficients["Coefficient"] = (
    linear_coefficients["Coefficient"]
    .round(4)
)

intercept = round(
    linear_model.intercept_,
    4
)

print("Intercept:")
print(intercept)

print("\nLinear Regression coefficients:")
display(linear_coefficients)

Intercept:
0.8417

Linear Regression coefficients:


,Feature,Coefficient
0,CON_score,0.1882
1,SAT_score,0.6078


In [10]:
# در این سلول VIF استاندارد برای CON و SAT با اضافه کردن Constant محاسبه می‌شود.

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

X_vif = sm.add_constant(
    X_train
)

vif_rows = []

for i in range(1, X_vif.shape[1]):

    vif_rows.append({
        "Feature": X_vif.columns[i],

        "VIF": variance_inflation_factor(
            X_vif.values,
            i
        )
    })

vif_table = pd.DataFrame(
    vif_rows
)

vif_table["VIF"] = (
    vif_table["VIF"]
    .round(3)
)

display(vif_table)

,Feature,VIF
0,CON_score,3.032
1,SAT_score,3.032


In [11]:
# در این سلول پایداری Linear Regression با 5-Fold Cross-Validation فقط روی داده آموزشی Dataset B بررسی می‌شود.

from sklearn.model_selection import KFold, cross_val_score

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# MAE در پنج Fold
cv_mae_scores = -cross_val_score(
    linear_model,
    X_train,
    y_train,
    cv=cv,
    scoring="neg_mean_absolute_error"
)

# MSE در پنج Fold
cv_mse_scores = -cross_val_score(
    linear_model,
    X_train,
    y_train,
    cv=cv,
    scoring="neg_mean_squared_error"
)

# تبدیل MSE به RMSE
cv_rmse_scores = np.sqrt(
    cv_mse_scores
)

# R² در پنج Fold
cv_r2_scores = cross_val_score(
    linear_model,
    X_train,
    y_train,
    cv=cv,
    scoring="r2"
)


print("Linear Regression - 5-Fold Cross-Validation")

print("\nMAE scores:")
print(np.round(cv_mae_scores, 4))

print(
    "Mean MAE:",
    round(cv_mae_scores.mean(), 4)
)

print(
    "Std MAE:",
    round(cv_mae_scores.std(), 4)
)


print("\nRMSE scores:")
print(np.round(cv_rmse_scores, 4))

print(
    "Mean RMSE:",
    round(cv_rmse_scores.mean(), 4)
)

print(
    "Std RMSE:",
    round(cv_rmse_scores.std(), 4)
)


print("\nR² scores:")
print(np.round(cv_r2_scores, 4))

print(
    "Mean R²:",
    round(cv_r2_scores.mean(), 4)
)

print(
    "Std R²:",
    round(cv_r2_scores.std(), 4)
)

Linear Regression - 5-Fold Cross-Validation

MAE scores:
[0.3541 0.2855 0.3454 0.3363 0.2657]
Mean MAE: 0.3174
Std MAE: 0.0352

RMSE scores:
[0.5301 0.4169 0.4891 0.5247 0.3999]
Mean RMSE: 0.4721
Std RMSE: 0.0542

R² scores:
[0.5658 0.6602 0.5629 0.6571 0.5873]
Mean R²: 0.6067
Std R²: 0.0433


In [12]:
# در این سلول بهترین تعداد همسایه برای مدل KNN با Cross-Validation فقط روی داده آموزشی انتخاب می‌شود.

from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV

knn_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsRegressor())
])

knn_parameters = {
    "knn__n_neighbors": list(range(3, 31))
}

knn_grid = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=knn_parameters,
    cv=cv,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

knn_grid.fit(
    X_train,
    y_train
)

best_k = knn_grid.best_params_[
    "knn__n_neighbors"
]

print("Best number of neighbors (K):")
print(best_k)

print("\nBest CV RMSE:")

best_cv_rmse = np.sqrt(
    -knn_grid.best_score_
)

print(
    round(best_cv_rmse, 4)
)

Best number of neighbors (K):
17

Best CV RMSE:
0.486


In [13]:
# در این سلول بهترین مدل KNN انتخاب‌شده روی داده Test مستقل ارزیابی می‌شود.

best_knn_model = knn_grid.best_estimator_

knn_predictions = best_knn_model.predict(
    X_test
)

knn_mae = mean_absolute_error(
    y_test,
    knn_predictions
)

knn_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        knn_predictions
    )
)

knn_r2 = r2_score(
    y_test,
    knn_predictions
)

print("KNN Regression performance:")

print(
    "MAE :",
    round(knn_mae, 4)
)

print(
    "RMSE:",
    round(knn_rmse, 4)
)

print(
    "R²  :",
    round(knn_r2, 4)
)

KNN Regression performance:
MAE : 0.3474
RMSE: 0.5054
R²  : 0.6329


In [14]:
# در این سلول عملکرد Baseline، Linear Regression و KNN در یک جدول مقایسه می‌شود.

model_comparison = pd.DataFrame({

    "Model": [
        "Baseline",
        "Linear Regression",
        "KNN"
    ],

    "MAE": [
        baseline_mae,
        linear_mae,
        knn_mae
    ],

    "RMSE": [
        baseline_rmse,
        linear_rmse,
        knn_rmse
    ],

    "R2": [
        baseline_r2,
        linear_r2,
        knn_r2
    ]
})

model_comparison[
    ["MAE", "RMSE", "R2"]
] = model_comparison[
    ["MAE", "RMSE", "R2"]
].round(4)

display(model_comparison)

,Model,MAE,RMSE,R2
0,Baseline,0.6460,0.8374,-0.0077
1,Linear Regression,0.3254,0.5315,0.5941
2,KNN,0.3474,0.5054,0.6329


In [15]:
# در این سلول بهترین تنظیمات Decision Tree فقط با Cross-Validation روی داده آموزشی انتخاب می‌شوند.

from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV

decision_tree = DecisionTreeRegressor(
    random_state=42
)

tree_parameters = {

    "max_depth": [
        2, 3, 4, 5, 6, None
    ],

    "min_samples_split": [
        2, 5, 10, 15
    ],

    "min_samples_leaf": [
        1, 2, 5, 10
    ]
}

tree_grid = GridSearchCV(
    estimator=decision_tree,
    param_grid=tree_parameters,
    cv=cv,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

tree_grid.fit(
    X_train,
    y_train
)

best_tree_model = tree_grid.best_estimator_

best_tree_cv_rmse = np.sqrt(
    -tree_grid.best_score_
)

print("Best Decision Tree parameters:")
print(tree_grid.best_params_)

print("\nBest Decision Tree CV RMSE:")
print(
    round(
        best_tree_cv_rmse,
        4
    )
)

Best Decision Tree parameters:
{'max_depth': 4, 'min_samples_leaf': 1, 'min_samples_split': 10}

Best Decision Tree CV RMSE:
0.4973


In [16]:
# در این سلول بهترین Decision Tree انتخاب‌شده روی داده Test مستقل ارزیابی می‌شود.

tree_predictions = best_tree_model.predict(
    X_test
)

tree_mae = mean_absolute_error(
    y_test,
    tree_predictions
)

tree_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        tree_predictions
    )
)

tree_r2 = r2_score(
    y_test,
    tree_predictions
)

print("Decision Tree Regression performance:")

print(
    "MAE :",
    round(tree_mae, 4)
)

print(
    "RMSE:",
    round(tree_rmse, 4)
)

print(
    "R²  :",
    round(tree_r2, 4)
)

Decision Tree Regression performance:
MAE : 0.3332
RMSE: 0.5473
R²  : 0.5695


In [17]:
# در این سلول بهترین تنظیمات Random Forest فقط با Cross-Validation روی داده آموزشی انتخاب می‌شوند.

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

random_forest = RandomForestRegressor(
    random_state=42
)

rf_parameters = {

    "n_estimators": [
        100,
        200,
        300
    ],

    "max_depth": [
        3,
        4,
        5,
        None
    ],

    "min_samples_split": [
        2,
        5,
        10
    ],

    "min_samples_leaf": [
        1,
        2,
        5
    ]
}

rf_grid = GridSearchCV(
    estimator=random_forest,
    param_grid=rf_parameters,
    cv=cv,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

rf_grid.fit(
    X_train,
    y_train
)

best_rf_model = rf_grid.best_estimator_

best_rf_cv_rmse = np.sqrt(
    -rf_grid.best_score_
)

print("Best Random Forest parameters:")
print(rf_grid.best_params_)

print("\nBest Random Forest CV RMSE:")
print(
    round(
        best_rf_cv_rmse,
        4
    )
)

Best Random Forest parameters:
{'max_depth': 3, 'min_samples_leaf': 1, 'min_samples_split': 10, 'n_estimators': 300}

Best Random Forest CV RMSE:
0.4851


In [18]:
# در این سلول بهترین تنظیمات مدل‌ها و عملکرد Cross-Validation آن‌ها در یک جدول جمع‌بندی و ذخیره می‌شود.

TABLES_FOLDER = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "tables"
)

os.makedirs(
    TABLES_FOLDER,
    exist_ok=True
)


model_tuning_summary = pd.DataFrame({

    "Model": [
        "Linear Regression",
        "KNN",
        "Decision Tree",
        "Random Forest"
    ],

    "Best_Parameters": [
        "Default / No hyperparameter tuning",

        f"n_neighbors={best_k}, scaling=StandardScaler",

        str(tree_grid.best_params_),

        str(rf_grid.best_params_)
    ],

    "CV_RMSE": [
        cv_rmse_scores.mean(),
        best_cv_rmse,
        best_tree_cv_rmse,
        best_rf_cv_rmse
    ]
})


model_tuning_summary["CV_RMSE"] = (
    model_tuning_summary["CV_RMSE"]
    .round(4)
)

display(model_tuning_summary)


tuning_table_path = os.path.join(
    TABLES_FOLDER,
    "Model_Tuning_Summary.xlsx"
)

model_tuning_summary.to_excel(
    tuning_table_path,
    index=False
)

print("\nModel tuning table saved:")
print(tuning_table_path)

,Model,Best_Parameters,CV_RMSE
0,Linear Regression,Default / No hyperparameter tuning,0.4721
1,KNN,"n_neighbors=17, scaling=StandardScaler",0.4860
2,Decision Tree,"{'max_depth': 4, 'min_samples_leaf': 1, 'min_s...",0.4973
3,Random Forest,"{'max_depth': 3, 'min_samples_leaf': 1, 'min_s...",0.4851



Model tuning table saved:
F:\E_Learning_Continuance_ML\outputs\tables\Model_Tuning_Summary.xlsx


In [19]:
# در این سلول بهترین Random Forest انتخاب‌شده روی داده Test مستقل ارزیابی می‌شود.

rf_predictions = best_rf_model.predict(
    X_test
)

rf_mae = mean_absolute_error(
    y_test,
    rf_predictions
)

rf_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        rf_predictions
    )
)

rf_r2 = r2_score(
    y_test,
    rf_predictions
)

print("Random Forest Regression performance:")

print(
    "MAE :",
    round(rf_mae, 4)
)

print(
    "RMSE:",
    round(rf_rmse, 4)
)

print(
    "R²  :",
    round(rf_r2, 4)
)

Random Forest Regression performance:
MAE : 0.339
RMSE: 0.5343
R²  : 0.5898


In [20]:
# در این سلول عملکرد تمام مدل‌های ارزیابی‌شده روی Test Set در یک جدول جمع‌بندی و ذخیره می‌شود.

model_test_performance = pd.DataFrame({

    "Model": [
        "Baseline",
        "Linear Regression",
        "KNN",
        "Decision Tree",
        "Random Forest"
    ],

    "MAE": [
        baseline_mae,
        linear_mae,
        knn_mae,
        tree_mae,
        rf_mae
    ],

    "RMSE": [
        baseline_rmse,
        linear_rmse,
        knn_rmse,
        tree_rmse,
        rf_rmse
    ],

    "R2": [
        baseline_r2,
        linear_r2,
        knn_r2,
        tree_r2,
        rf_r2
    ]
})

model_test_performance[
    ["MAE", "RMSE", "R2"]
] = model_test_performance[
    ["MAE", "RMSE", "R2"]
].round(4)

display(model_test_performance)


performance_path = os.path.join(
    TABLES_FOLDER,
    "Model_Test_Performance.xlsx"
)

model_test_performance.to_excel(
    performance_path,
    index=False
)

print("\nModel performance table saved:")
print(performance_path)

,Model,MAE,RMSE,R2
0,Baseline,0.6460,0.8374,-0.0077
1,Linear Regression,0.3254,0.5315,0.5941
2,KNN,0.3474,0.5054,0.6329
3,Decision Tree,0.3332,0.5473,0.5695
4,Random Forest,0.3390,0.5343,0.5898



Model performance table saved:
F:\E_Learning_Continuance_ML\outputs\tables\Model_Test_Performance.xlsx


In [21]:
# در این سلول تمام مدل‌های فعلی با همان 5-Fold Cross-Validation و معیارهای MAE، RMSE و R² مقایسه می‌شوند.

from sklearn.model_selection import cross_validate

models_for_cv = {
    "Linear Regression": linear_model,
    "KNN": best_knn_model,
    "Decision Tree": best_tree_model,
    "Random Forest": best_rf_model
}

cv_results_rows = []

scoring = {
    "MAE": "neg_mean_absolute_error",
    "MSE": "neg_mean_squared_error",
    "R2": "r2"
}

for model_name, model in models_for_cv.items():

    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    mae_scores = -scores["test_MAE"]
    rmse_scores = np.sqrt(
        -scores["test_MSE"]
    )
    r2_scores = scores["test_R2"]

    cv_results_rows.append({

        "Model": model_name,

        "CV_MAE_Mean": mae_scores.mean(),
        "CV_MAE_Std": mae_scores.std(),

        "CV_RMSE_Mean": rmse_scores.mean(),
        "CV_RMSE_Std": rmse_scores.std(),

        "CV_R2_Mean": r2_scores.mean(),
        "CV_R2_Std": r2_scores.std()
    })


cv_model_comparison = pd.DataFrame(
    cv_results_rows
)

numeric_columns = [
    "CV_MAE_Mean",
    "CV_MAE_Std",
    "CV_RMSE_Mean",
    "CV_RMSE_Std",
    "CV_R2_Mean",
    "CV_R2_Std"
]

cv_model_comparison[numeric_columns] = (
    cv_model_comparison[numeric_columns]
    .round(4)
)

cv_model_comparison = (
    cv_model_comparison
    .sort_values(
        by="CV_RMSE_Mean"
    )
    .reset_index(drop=True)
)

display(cv_model_comparison)

,Model,CV_MAE_Mean,CV_MAE_Std,CV_RMSE_Mean,CV_RMSE_Std,CV_R2_Mean,CV_R2_Std
0,Linear Regression,0.3174,0.0352,0.4721,0.0542,0.6067,0.0433
1,KNN,0.3192,0.0373,0.4808,0.0708,0.5951,0.0479
2,Random Forest,0.3103,0.0296,0.4821,0.0539,0.5898,0.0426
3,Decision Tree,0.3079,0.0350,0.4939,0.0584,0.5692,0.0555


In [22]:
# در این سلول نتایج Cross-Validation تمام مدل‌های فعلی برای استفاده در مقاله ذخیره می‌شوند.

cv_performance_path = os.path.join(
    TABLES_FOLDER,
    "Model_Cross_Validation_Performance.xlsx"
)

cv_model_comparison.to_excel(
    cv_performance_path,
    index=False
)

print("Cross-validation performance table saved:")
print(cv_performance_path)

Cross-validation performance table saved:
F:\E_Learning_Continuance_ML\outputs\tables\Model_Cross_Validation_Performance.xlsx


In [23]:
# در این سلول بهترین تنظیمات Gradient Boosting فقط با Cross-Validation روی داده آموزشی انتخاب می‌شوند.

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV

gradient_boosting = GradientBoostingRegressor(
    random_state=42
)

gb_parameters = {

    "n_estimators": [
        50,
        100,
        200
    ],

    "learning_rate": [
        0.01,
        0.05,
        0.1
    ],

    "max_depth": [
        1,
        2,
        3
    ],

    "min_samples_leaf": [
        1,
        3,
        5
    ]
}

gb_grid = GridSearchCV(
    estimator=gradient_boosting,
    param_grid=gb_parameters,
    cv=cv,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

gb_grid.fit(
    X_train,
    y_train
)

best_gb_model = gb_grid.best_estimator_

best_gb_cv_rmse = np.sqrt(
    -gb_grid.best_score_
)

print("Best Gradient Boosting parameters:")
print(gb_grid.best_params_)

print("\nBest Gradient Boosting CV RMSE:")
print(
    round(
        best_gb_cv_rmse,
        4
    )
)

Best Gradient Boosting parameters:
{'learning_rate': 0.05, 'max_depth': 2, 'min_samples_leaf': 1, 'n_estimators': 50}

Best Gradient Boosting CV RMSE:
0.4871


In [24]:
# در این سلول Gradient Boosting منتخب با همان روش 5-Fold Cross-Validation مدل‌های قبلی ارزیابی می‌شود.

gb_cv_scores = cross_validate(
    best_gb_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

gb_cv_mae = -gb_cv_scores["test_MAE"]

gb_cv_rmse = np.sqrt(
    -gb_cv_scores["test_MSE"]
)

gb_cv_r2 = gb_cv_scores["test_R2"]


print("Gradient Boosting - 5-Fold Cross-Validation")

print(
    "Mean MAE:",
    round(gb_cv_mae.mean(), 4)
)

print(
    "Std MAE:",
    round(gb_cv_mae.std(), 4)
)

print(
    "Mean RMSE:",
    round(gb_cv_rmse.mean(), 4)
)

print(
    "Std RMSE:",
    round(gb_cv_rmse.std(), 4)
)

print(
    "Mean R²:",
    round(gb_cv_r2.mean(), 4)
)

print(
    "Std R²:",
    round(gb_cv_r2.std(), 4)
)

Gradient Boosting - 5-Fold Cross-Validation
Mean MAE: 0.3277
Std MAE: 0.031
Mean RMSE: 0.4832
Std RMSE: 0.0614
Mean R²: 0.5903
Std R²: 0.0345


In [25]:
# در این سلول Gradient Boosting منتخب روی Test Set مستقل Dataset B ارزیابی می‌شود.

gb_predictions = best_gb_model.predict(
    X_test
)

gb_mae = mean_absolute_error(
    y_test,
    gb_predictions
)

gb_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        gb_predictions
    )
)

gb_r2 = r2_score(
    y_test,
    gb_predictions
)

print("Gradient Boosting Regression performance:")

print(
    "MAE :",
    round(gb_mae, 4)
)

print(
    "RMSE:",
    round(gb_rmse, 4)
)

print(
    "R²  :",
    round(gb_r2, 4)
)

Gradient Boosting Regression performance:
MAE : 0.359
RMSE: 0.5351
R²  : 0.5885


In [26]:
# در این سلول نتایج Cross-Validation پنج مدل در یک جدول نهایی جمع‌بندی و ذخیره می‌شوند.

cv_model_comparison_final = pd.DataFrame({

    "Model": [
        "Linear Regression",
        "KNN",
        "Random Forest",
        "Gradient Boosting",
        "Decision Tree"
    ],

    "CV_MAE_Mean": [
        cv_mae_scores.mean(),
        0.3192,
        0.3103,
        gb_cv_mae.mean(),
        0.3079
    ],

    "CV_MAE_Std": [
        cv_mae_scores.std(),
        0.0373,
        0.0296,
        gb_cv_mae.std(),
        0.0350
    ],

    "CV_RMSE_Mean": [
        cv_rmse_scores.mean(),
        0.4808,
        0.4821,
        gb_cv_rmse.mean(),
        0.4939
    ],

    "CV_RMSE_Std": [
        cv_rmse_scores.std(),
        0.0708,
        0.0539,
        gb_cv_rmse.std(),
        0.0584
    ],

    "CV_R2_Mean": [
        cv_r2_scores.mean(),
        0.5951,
        0.5898,
        gb_cv_r2.mean(),
        0.5692
    ],

    "CV_R2_Std": [
        cv_r2_scores.std(),
        0.0479,
        0.0426,
        gb_cv_r2.std(),
        0.0555
    ]
})

numeric_columns = [
    "CV_MAE_Mean",
    "CV_MAE_Std",
    "CV_RMSE_Mean",
    "CV_RMSE_Std",
    "CV_R2_Mean",
    "CV_R2_Std"
]

cv_model_comparison_final[numeric_columns] = (
    cv_model_comparison_final[numeric_columns]
    .round(4)
)

cv_model_comparison_final = (
    cv_model_comparison_final
    .sort_values("CV_RMSE_Mean")
    .reset_index(drop=True)
)

display(cv_model_comparison_final)

cv_model_comparison_final.to_excel(
    os.path.join(
        TABLES_FOLDER,
        "Model_Cross_Validation_Performance.xlsx"
    ),
    index=False
)

print("Final cross-validation table saved.")

,Model,CV_MAE_Mean,CV_MAE_Std,CV_RMSE_Mean,CV_RMSE_Std,CV_R2_Mean,CV_R2_Std
0,Linear Regression,0.3174,0.0352,0.4721,0.0542,0.6067,0.0433
1,KNN,0.3192,0.0373,0.4808,0.0708,0.5951,0.0479
2,Random Forest,0.3103,0.0296,0.4821,0.0539,0.5898,0.0426
3,Gradient Boosting,0.3277,0.0310,0.4832,0.0614,0.5903,0.0345
4,Decision Tree,0.3079,0.0350,0.4939,0.0584,0.5692,0.0555


Final cross-validation table saved.


In [27]:
# در این سلول عملکرد نهایی پنج مدل و Baseline روی Test Set جمع‌بندی و ذخیره می‌شود.

model_test_performance_final = pd.DataFrame({

    "Model": [
        "Baseline",
        "Linear Regression",
        "KNN",
        "Decision Tree",
        "Random Forest",
        "Gradient Boosting"
    ],

    "MAE": [
        baseline_mae,
        linear_mae,
        knn_mae,
        tree_mae,
        rf_mae,
        gb_mae
    ],

    "RMSE": [
        baseline_rmse,
        linear_rmse,
        knn_rmse,
        tree_rmse,
        rf_rmse,
        gb_rmse
    ],

    "R2": [
        baseline_r2,
        linear_r2,
        knn_r2,
        tree_r2,
        rf_r2,
        gb_r2
    ]
})

model_test_performance_final[
    ["MAE", "RMSE", "R2"]
] = model_test_performance_final[
    ["MAE", "RMSE", "R2"]
].round(4)

display(model_test_performance_final)

model_test_performance_final.to_excel(
    os.path.join(
        TABLES_FOLDER,
        "Model_Test_Performance.xlsx"
    ),
    index=False
)

print("Final test-performance table saved.")

,Model,MAE,RMSE,R2
0,Baseline,0.6460,0.8374,-0.0077
1,Linear Regression,0.3254,0.5315,0.5941
2,KNN,0.3474,0.5054,0.6329
3,Decision Tree,0.3332,0.5473,0.5695
4,Random Forest,0.3390,0.5343,0.5898
5,Gradient Boosting,0.3590,0.5351,0.5885


Final test-performance table saved.


In [28]:
# در این سلول تنظیمات نهایی مدل‌های موردبررسی برای مستندسازی پژوهش ذخیره می‌شوند.

model_tuning_summary_final = pd.DataFrame({

    "Model": [
        "Linear Regression",
        "KNN",
        "Decision Tree",
        "Random Forest",
        "Gradient Boosting"
    ],

    "Best_Parameters": [
        "Default / No hyperparameter tuning",

        f"n_neighbors={best_k}, scaling=StandardScaler",

        str(tree_grid.best_params_),

        str(rf_grid.best_params_),

        str(gb_grid.best_params_)
    ]
})

display(model_tuning_summary_final)

model_tuning_summary_final.to_excel(
    os.path.join(
        TABLES_FOLDER,
        "Model_Tuning_Summary.xlsx"
    ),
    index=False
)

print("Final tuning table saved.")

,Model,Best_Parameters
0,Linear Regression,Default / No hyperparameter tuning
1,KNN,"n_neighbors=17, scaling=StandardScaler"
2,Decision Tree,"{'max_depth': 4, 'min_samples_leaf': 1, 'min_s..."
3,Random Forest,"{'max_depth': 3, 'min_samples_leaf': 1, 'min_s..."
4,Gradient Boosting,"{'learning_rate': 0.05, 'max_depth': 2, 'min_s..."


Final tuning table saved.


In [29]:
# در این سلول مدل نهایی Linear Regression پس از پایان انتخاب مدل، روی تمام داده‌های Dataset B آموزش داده می‌شود.

from sklearn.linear_model import LinearRegression

final_linear_model = LinearRegression()

final_linear_model.fit(
    X,
    y
)

print("Final Linear Regression model fitted on full Dataset B.")

print("\nNumber of development samples:")
print(len(X))

print("\nFeatures:")
print(feature_columns)

print("\nIntercept:")
print(round(final_linear_model.intercept_, 4))

print("\nCoefficients:")

for feature, coefficient in zip(
    feature_columns,
    final_linear_model.coef_
):
    print(
        feature,
        ":",
        round(coefficient, 4)
    )

Final Linear Regression model fitted on full Dataset B.

Number of development samples:
368

Features:
['CON_score', 'SAT_score']

Intercept:
0.8899

Coefficients:
CON_score : 0.1917
SAT_score : 0.593


In [30]:
# در این سلول مدل نهایی آموزش‌دیده برای استفاده مستقیم در External Validation ذخیره می‌شود.

import joblib

MODELS_FOLDER = os.path.join(
    PROJECT_ROOT,
    "models"
)

os.makedirs(
    MODELS_FOLDER,
    exist_ok=True
)

final_model_path = os.path.join(
    MODELS_FOLDER,
    "Final_Linear_Regression_Model.joblib"
)

joblib.dump(
    final_linear_model,
    final_model_path
)

print("Final model saved:")
print(final_model_path)

Final model saved:
F:\E_Learning_Continuance_ML\models\Final_Linear_Regression_Model.joblib


In [31]:
# در این سلول ضرایب و عرض از مبدأ مدل نهایی Linear Regression در یک جدول ذخیره می‌شوند.

final_coefficients = pd.DataFrame({

    "Term": [
        "Intercept",
        "CON_score",
        "SAT_score"
    ],

    "Coefficient": [
        final_linear_model.intercept_,
        final_linear_model.coef_[0],
        final_linear_model.coef_[1]
    ]
})

final_coefficients["Coefficient"] = (
    final_coefficients["Coefficient"]
    .round(4)
)

display(final_coefficients)


coefficients_path = os.path.join(
    TABLES_FOLDER,
    "Final_Linear_Regression_Coefficients.xlsx"
)

final_coefficients.to_excel(
    coefficients_path,
    index=False
)

print("\nFinal coefficients saved:")
print(coefficients_path)

,Term,Coefficient
0,Intercept,0.8899
1,CON_score,0.1917
2,SAT_score,0.5930



Final coefficients saved:
F:\E_Learning_Continuance_ML\outputs\tables\Final_Linear_Regression_Coefficients.xlsx


In [32]:
# در این سلول مدل ذخیره‌شده دوباره بارگذاری می‌شود تا صحت ذخیره‌سازی آن بررسی شود.

loaded_final_model = joblib.load(
    final_model_path
)

original_predictions = final_linear_model.predict(
    X
)

loaded_predictions = loaded_final_model.predict(
    X
)

max_prediction_difference = np.max(
    np.abs(
        original_predictions
        - loaded_predictions
    )
)

print("Loaded model type:")
print(type(loaded_final_model).__name__)

print("\nFeature order:")
print(feature_columns)

print("\nNumber of development samples:")
print(len(X))

print("\nMaximum prediction difference:")
print(max_prediction_difference)

Loaded model type:
LinearRegression

Feature order:
['CON_score', 'SAT_score']

Number of development samples:
368

Maximum prediction difference:
0.0


In [33]:
# در این سلول مشخصات مدل نهایی برای مستندسازی و بازتولیدپذیری پژوهش ذخیره می‌شود.

import json

model_metadata = {

    "model_name": "Linear Regression",

    "development_dataset": "Dataset B",

    "development_sample_size": int(len(X)),

    "predictors": [
        "CON_score",
        "SAT_score"
    ],

    "target": "CI_score",

    "model_selection_metric": "5-Fold CV RMSE",

    "selected_cv_rmse": 0.4721,

    "external_validation_dataset": "EFL",

    "model_file": "Final_Linear_Regression_Model.joblib"
}


metadata_path = os.path.join(
    MODELS_FOLDER,
    "Final_Model_Metadata.json"
)


with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        model_metadata,
        file,
        indent=4,
        ensure_ascii=False
    )


print("Final model metadata saved:")
print(metadata_path)

print("\nModel metadata:")
print(
    json.dumps(
        model_metadata,
        indent=4,
        ensure_ascii=False
    )
)

Final model metadata saved:
F:\E_Learning_Continuance_ML\models\Final_Model_Metadata.json

Model metadata:
{
    "model_name": "Linear Regression",
    "development_dataset": "Dataset B",
    "development_sample_size": 368,
    "predictors": [
        "CON_score",
        "SAT_score"
    ],
    "target": "CI_score",
    "model_selection_metric": "5-Fold CV RMSE",
    "selected_cv_rmse": 0.4721,
    "external_validation_dataset": "EFL",
    "model_file": "Final_Linear_Regression_Model.joblib"
}
